# Belle II Continuum Suppression

Binary signal-versus-continuum classification using the instructor's Belle II CSV. The private data is not committed.

## 1. Setup and reproducibility

In [ ]:
from pathlib import Path
import importlib.util
import sys
import pandas as pd
from IPython.display import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baselines import evaluate_baselines
from src.config import (
    DB_PATH, RANDOM_STATE, RAW_DATA_DIR, TEST_SIZE, VALIDATION_SIZE
)
from src.data_io import (
    assign_split, audit_frame, load_raw_to_db, prepare_professor_binary
)
from src.db import read_table, table_exists, update_cell, write_table
from src.eda import (class_balance, feature_columns, feature_correlations,
                     feature_separation, plot_class_balance,
                     plot_correlation_heatmap, plot_feature_distributions,
                     training_events)
from src.cnn import fit_cnn_validation
from src.features import analyze_feature_reduction, plot_pca_variance
from src.models import (fit_ankit_validation, fit_koustav_mlp_validation,
                        fit_vishruti_validation)

print(f"Project root: {PROJECT_ROOT}")
print(
    f"Random state: {RANDOM_STATE}; split fractions: "
    f"train={1 - VALIDATION_SIZE - TEST_SIZE:.0%}, "
    f"validation={VALIDATION_SIZE:.0%}, test={TEST_SIZE:.0%}"
)

## 2. Instructor dataset

Use `data_hep - data_hep.csv` only. Its process flags are converted to the binary target below.

The original CSV remains private and unchanged.

## 3. Locate the instructor CSV

In [ ]:
raw_csv_path = RAW_DATA_DIR / "data_hep - data_hep.csv"
if not raw_csv_path.is_file():
    raw_csv_path = None

if raw_csv_path is None:
    print("Waiting for data/raw/data_hep - data_hep.csv.")
else:
    print(f"Dataset found: {raw_csv_path.name}")

## 4. Store the untouched CSV in SQLite

In [ ]:
raw_events = None
if raw_csv_path is not None:
    if table_exists("raw_events", DB_PATH):
        raw_events = read_table("raw_events", DB_PATH)
        print("Read the existing immutable raw_events table.")
    else:
        raw_events = load_raw_to_db(raw_csv_path, DB_PATH)
        print("Wrote the immutable raw_events table.")
    print(f"Loaded {len(raw_events):,} rows and {raw_events.shape[1]} columns.")
else:
    print("Ingestion skipped until exactly one source CSV is available.")

## 5. Audit the raw data

In [ ]:
raw_audit = audit_frame(raw_events) if raw_events is not None else None
if raw_audit is not None:
    print(f"Rows: {raw_audit['rows']:,}; columns: {raw_audit['columns']}")
    print(f"Duplicate rows: {raw_audit['duplicate_rows']:,}")
    print(f"Constant columns: {raw_audit['constant_columns']}")
    print(f"Six-way type counts: {raw_events['type'].value_counts().sort_index().to_dict()}")
    print(f"Missing cells: {sum(raw_audit['missing_counts'].values()):,}")
    print(f"Non-finite cells: {sum(raw_audit['non_finite_counts'].values()):,}")
    display(raw_events.head())
else:
    print("Audit skipped because no dataset has been loaded.")

## 6. Prepare the binary label

`prepare_professor_binary` applies the instructor's process mapping: 0 = charged B pair, 1 = neutral B pair, 2 = $c\bar c$, 3 = $u\bar u$, 4 = $d\bar d$, 5 = $s\bar s$. It combines 0/1 as signal (`target=1`) and 2/3/4/5 as continuum background (`target=0`). The original `type` and exported row number are not model inputs.

In [ ]:
prepared_events = prepare_professor_binary(raw_events) if raw_events is not None else None
if prepared_events is not None:
    print(f"Rows retained: {len(prepared_events):,}/{len(raw_events):,}")
    print("Excluded from prepared data: Unnamed: 0 and six-way type")
    display(prepared_events['target'].value_counts().sort_index().rename('rows'))
else:
    print("Binary preparation waits for the professor CSV.")

## 7. Store the 70/15/15 train/validation/test split

Split by row position, so duplicate pandas index labels cannot overlap assignments.

In [ ]:
events_with_split = None
if prepared_events is not None:
    if table_exists("cleaned_events", DB_PATH):
        events_with_split = read_table("cleaned_events", DB_PATH)
        print("Read persisted cleaned_events; did not re-split.")
    else:
        events_with_split = assign_split(
            prepared_events, "target",
            validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
        )
        write_table(events_with_split, "cleaned_events", db_path=DB_PATH)
        print("Wrote cleaned_events with one persisted split.")
    display(events_with_split.groupby(["split", "target"]).size().rename("rows"))
else:
    print("Split assignment waits for professor data.")

## 8. Training-only EDA and class balance

Explore training rows only; validation and test remain held out. Raw `type` and `Unnamed: 0` columns are rejected. Counts are unweighted.

In [ ]:
train_events = training_events(events_with_split) if events_with_split is not None else None
if train_events is not None:
    physics_features = feature_columns(train_events)
    print(f'Training rows: {len(train_events):,}; features: {len(physics_features)}')
    balance = class_balance(train_events)
    display(balance)
    display(plot_class_balance(balance))
else:
    print('EDA waits for the instructor dataset.')

## 9. Feature distributions and separation

Plot representative event-shape variables; rank all physics features by single-feature ROC-AUC. This ranking is not a final model result.

In [ ]:
separation = None
if train_events is not None:
    display(plot_feature_distributions(train_events,
           ['R2', 'thrust', 'thrustBm', 'foxWolframR2']))
    separation = feature_separation(train_events)
    display(separation.head(15))
else:
    print('Feature EDA waits for training rows.')

## 10. Correlations and R2 physics check

Check whether signal has lower R2 than continuum; investigate any reversal before modeling.

In [ ]:
if train_events is not None:
    display(plot_correlation_heatmap(feature_correlations(train_events)))
    r2_summary = separation.loc[separation['feature'] == 'R2'].iloc[0]
    print(f"R2 median: signal={r2_summary['signal_median']:.4f}, background={r2_summary['background_median']:.4f}")
    print('Expected R2 direction?', r2_summary['signal_median'] < r2_summary['background_median'])
else:
    print('Physics check waits for training rows.')

## 11. SQLite read, write, and modify

Store and update only the derived training EDA summary; raw and cleaned events stay unchanged.

In [ ]:
if separation is not None:
    eda_summary = separation.assign(status='computed')
    write_table(eda_summary, 'eda_feature_summary', if_exists='replace', db_path=DB_PATH)
    top_feature = eda_summary.iloc[0]['feature']
    changed = update_cell('eda_feature_summary', 'status', 'candidate_for_review',
                          'feature', top_feature, db_path=DB_PATH)
    print(f'Updated {changed} derived row(s): {top_feature}')
    display(read_table('eda_feature_summary', DB_PATH).head(10))
else:
    print('Derived SQLite summary waits for EDA.')

## 12. Training-defined baselines

Compare a training-majority prediction and a training-selected R2 cut on validation. Test rows remain untouched.

In [ ]:
baseline_definition, baseline_validation_results = (None, None)
if events_with_split is not None:
    baseline_definition, baseline_validation_results = evaluate_baselines(events_with_split)
    print(f'Training-selected R2 cut: {baseline_definition.r2_cut:.6f}')
    display(baseline_validation_results)
else:
    print('Baselines wait for the instructor data.')

## 13. Ankit's full-feature models

Train Logistic Regression and Decision Tree on stored training rows; show provisional validation metrics only. The final test split stays untouched.

In [ ]:
ankit_fitted_models, ankit_validation_results = (None, None)
if events_with_split is not None:
    ankit_fitted_models, ankit_validation_results = fit_ankit_validation(events_with_split)
    display(ankit_validation_results)
    print('Provisional validation metrics; final test rows are untouched.')
else:
    print('Modeling waits for the instructor data.')

## 14. Vishruti's full-feature models

Train a scaled Linear SVM and histogram Gradient Boosting BDT; compare all implemented methods on validation only.

In [ ]:
vishruti_fitted_models, vishruti_validation_results = (None, None)
if events_with_split is not None:
    vishruti_fitted_models, vishruti_validation_results = fit_vishruti_validation(events_with_split)
    provisional_validation_table = pd.concat(
        [baseline_validation_results, ankit_validation_results, vishruti_validation_results],
        ignore_index=True,
    )
    display(provisional_validation_table.sort_values('roc_auc', ascending=False))
    print('Validation-only comparison; final test rows remain untouched.')
else:
    print('Vishruti models wait for the instructor data.')

## 15. Training-only feature reduction

Prune correlations above 0.95, inspect standardized PCA at 95% cumulative variance, rank tree importance, and freeze a 20-feature Logistic Regression RFE subset. Validation and test values do not influence this analysis.

In [ ]:
reduction = None
if events_with_split is not None:
    reduction = analyze_feature_reduction(events_with_split)
    print(f'Correlation pruning: {len(reduction.correlation_kept)} kept, {len(reduction.correlation_dropped)} dropped')
    print(f'PCA components for at least 95% training variance: {reduction.pca_components_95}')
    display(reduction.tree_importance.head(15))
    display(reduction.rfe_ranking.head(25))
    display(plot_pca_variance(reduction))
else:
    print('Feature reduction waits for the instructor data.')

## 16. Koustav's MLP

Compare the same scaled feed-forward neural network on all physics features and the frozen RFE subset. Its early-stopping partition is drawn only from training rows.

In [ ]:
mlp_full_model, mlp_full_results = (None, None)
mlp_reduced_model, mlp_reduced_results = (None, None)
if reduction is not None:
    mlp_full_model, mlp_full_results = fit_koustav_mlp_validation(events_with_split)
    mlp_reduced_model, mlp_reduced_results = fit_koustav_mlp_validation(
        events_with_split, reduction.selected_features
    )
    display(pd.concat([mlp_full_results, mlp_reduced_results], ignore_index=True))
else:
    print('MLP validation waits for the training-only reduction result.')

## 17. Experimental physics-grouped CNN

Convolution is used only for ordered CLEO-cone and KSFW-moment families; remaining scalar measurements use a dense branch. PyTorch is optional so the core notebook remains runnable without the large dependency.

In [ ]:
cnn_model, cnn_validation_results = (None, None)
if events_with_split is None:
    print('CNN validation waits for the instructor data.')
elif importlib.util.find_spec('torch') is None:
    print('CNN skipped: install the optional dependency with python -m pip install -r requirements-cnn.txt')
else:
    cnn_model, cnn_validation_results = fit_cnn_validation(events_with_split)
    display(cnn_validation_results)

## 18. Provisional validation comparison

This table supports model and feature-set selection. It is not a final result because the held-out test split remains unopened.

In [ ]:
if mlp_full_results is not None:
    validation_parts = [provisional_validation_table, mlp_full_results, mlp_reduced_results]
    if cnn_validation_results is not None:
        validation_parts.append(cnn_validation_results)
    koustav_validation_table = pd.concat(validation_parts, ignore_index=True)
    display(koustav_validation_table.sort_values('roc_auc', ascending=False))
    print('Freeze the final protocol before evaluating the test split.')
else:
    print('Combined comparison waits for Koustav model results.')

## 19. Frozen final test artifacts

The test evaluator is deliberately not called by this notebook. After reviewing Decision 24, run `python -m scripts.run_final_evaluation` exactly once. This cell displays its guarded outputs on later notebook runs.

In [ ]:
final_metrics_path = PROJECT_ROOT / 'results/tables/final_test_metrics.csv'
final_roc_path = PROJECT_ROOT / 'results/figures/final_test_roc.png'
final_confusion_path = PROJECT_ROOT / 'results/figures/final_test_confusion_matrices.png'
if final_metrics_path.is_file():
    display(pd.read_csv(final_metrics_path))
    display(Image(filename=str(final_roc_path)))
    display(Image(filename=str(final_confusion_path)))
else:
    print('Final test remains unopened. Review the frozen protocol, then run python -m scripts.run_final_evaluation once.')

## 20. Final interpretation

The BDT is strongest on the frozen test (ROC-AUC 0.9457). RFE reduces the MLP from 59 to 20 measured features with a small ROC-AUC decrease from 0.9416 to 0.9399, so reduced performance is comparable, not better. The experimental CNN reaches 0.9273 and does not beat the BDT or MLPs. These results come from one fixed split of simulated instructor data and do not establish detector-data or cross-dataset generalization.

## 21. Post-hoc reduced Random Forest

One Random Forest uses the frozen RFE-20 feature set and five-fold out-of-fold predictions on development rows only. It never accesses original test rows and is not an addition to the frozen headline ranking. Run `python -m scripts.run_posthoc_extensions` separately to create the artifacts displayed below.

In [ ]:
posthoc_rf_path = PROJECT_ROOT / 'results/tables/posthoc_reduced_random_forest_cv.csv'
posthoc_rf_diagnostics = PROJECT_ROOT / 'results/figures/posthoc_reduced_random_forest_diagnostics.png'
if posthoc_rf_path.is_file():
    display(pd.read_csv(posthoc_rf_path))
    display(Image(filename=str(posthoc_rf_diagnostics)))
else:
    print('Post-hoc model has not run. Use python -m scripts.run_posthoc_extensions; frozen test results remain unchanged.')

### Reduced Random Forest interpretation

Across 60,015 development-only out-of-fold predictions, the RFE-20 Random Forest reaches accuracy 0.8595, precision 0.8513, recall 0.8756, F1 0.8633, and ROC-AUC 0.9332. This demonstrates useful separation using 20 frozen measurements. It must not be ranked directly against models measured on the frozen test set because the evaluation protocols differ.

## 22. Post-hoc full versus reduced BDT

Because the histogram BDT was the strongest frozen-test family, we compare its unchanged configuration on all 59 features and the frozen RFE-20 subset. Both rows use identical five-fold out-of-fold predictions on development data only; original test rows remain excluded. Run `python -m scripts.run_reduced_bdt_comparison` separately to create the artifacts.

In [ ]:
posthoc_bdt_path = PROJECT_ROOT / 'results/tables/posthoc_bdt_feature_comparison_cv.csv'
posthoc_bdt_diagnostics = PROJECT_ROOT / 'results/figures/posthoc_bdt_feature_comparison.png'
if posthoc_bdt_path.is_file():
    display(pd.read_csv(posthoc_bdt_path))
    display(Image(filename=str(posthoc_bdt_diagnostics)))
else:
    print('Post-hoc BDT comparison has not run. Use python -m scripts.run_reduced_bdt_comparison; frozen test results remain unchanged.')

### BDT feature-reduction interpretation

Across 60,015 matched development-only out-of-fold predictions, the full BDT reaches accuracy 0.8773, F1 0.8804, and ROC-AUC 0.9481. The RFE-20 BDT reaches accuracy 0.8690, F1 0.8726, and ROC-AUC 0.9419. Removing 39 measurements costs about 0.0083 accuracy, 0.0079 F1, and 0.0062 ROC-AUC while retaining most discrimination. This is a simplicity/performance trade-off, not evidence that reduction improves the BDT, and neither row is a new held-out-test result.